# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two findings and my methodology questions

*Note: I could not locate the FlyRank research paper, so I audit two findings from my own
capstone, using the same questions: where does the label come from, and does the validation
design carry the claim?*

### Finding A: pages where one query dominates visibility declined more often
Held-out clients: 69.2% of flagged pages declined against 60.0% of other pages.
- **Where does the label come from?** `is_declining` is computed from the same daily table as the
  features: last-30-day impressions below 80% of the previous 30 days. The query features
  (`top_query_share` and others) come from a 90-day table whose window includes the last 30 days,
  so they overlap the period that defines the label. This makes the finding a same-period
  association, not a forecast.
- **Does the validation design carry the claim?** Partly. Cut-offs were learned on one half of the
  clients and applied to the other half (25 clients), which guards against client-specific
  patterns. But I chose the reason codes after inspecting the full data, and one split of 25
  clients is noisy.
- **Question a reviewer should ask:** Does the pattern hold when the query features are rebuilt
  from a window that ends before the label window starts?

### Finding B: query features carry more signal than impression volume
Under per-client splits, ROC-AUC was 0.496 for `imp_prev30` alone, 0.589 for the query features
alone and 0.635 for all five.
- **Where does the label come from?** The same `is_declining` rule. `imp_prev30` feeds the label,
  so I tested it alone to check the model was not recovering the label's arithmetic. It scored
  chance level.
- **Does the validation design carry the claim?** Partly. The splits were grouped by client, but
  only three seeds were run, and AUC varied by about 0.03 between seeds, so differences under
  that size are not reliable. The all-five versus query-only gap (0.046) is only slightly above
  that noise.
- **Question a reviewer should ask:** Does the ordering of the three feature sets stay the same
  across more seeds and a time-based split?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import os, getpass, duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
DATA_PATH = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'

df = con.sql(f"""
WITH b AS (SELECT MAX(report_date) AS end_d FROM read_parquet('{DATA_PATH}'))
SELECT f.client_hash_id, f.content_hash_id,
  SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
  SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
  SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
  AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prev30,
  STDDEV(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)    AS pos_std_prev30
FROM read_parquet('{DATA_PATH}') f, b
WHERE f.report_date > b.end_d - INTERVAL 60 DAY
GROUP BY 1, 2
HAVING imp_prev30 >= 100
""").df()

df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)
df['ctr_prev'] = df['clk_prev30'] / df['imp_prev30']
feats = ['imp_prev30', 'pos_prev30', 'pos_std_prev30', 'ctr_prev']
df = df.dropna(subset=feats).reset_index(drop=True)
X, y, g = df[feats], df['is_declining'], df['client_hash_id']
print('pages:', len(df), '  decline rate:', round(y.mean(), 3))

def make_model(seed):
    return RandomForestClassifier(n_estimators=60, max_depth=12, min_samples_leaf=50,
                                  random_state=seed, n_jobs=-1)

rows = []
for seed in range(3):
    splits = {
        'random': train_test_split(range(len(df)), test_size=0.25, random_state=seed, stratify=y),
        'per-client': next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(X, y, g)),
    }
    for name, (tr, te) in splits.items():
        m = make_model(seed).fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        base = max(y.iloc[te].mean(), 1 - y.iloc[te].mean())
        rows.append((name, seed, accuracy_score(y.iloc[te], p > 0.5), base, roc_auc_score(y.iloc[te], p)))

res = pd.DataFrame(rows, columns=['split', 'seed', 'accuracy', 'baseline', 'roc_auc'])
print(res.groupby('split')[['accuracy', 'baseline', 'roc_auc']].mean().round(3).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 111240   decline rate: 0.643
            accuracy  baseline  roc_auc
split                                  
per-client     0.638     0.637    0.619
random         0.660     0.643    0.649


## 2. My model under an honest split (before and after)

**Model.** A small random forest (60 trees, depth 12) predicting `is_declining` from four
features measured only in the earlier 30-day window: impressions, average position, position
volatility and click-through rate. Using only the earlier window keeps the features from
overlapping the label window. 111,240 pages; decline rate 0.643.

**Before (random split).** Accuracy 0.660 against a majority-class baseline of 0.643;
ROC-AUC 0.649.

**After (split grouped by client, so no client is in both train and test).** Accuracy 0.638
against a baseline of 0.637; ROC-AUC 0.619.

**Reading.** Under the random split the model beats the baseline by about 1.7 points of accuracy.
Once whole clients are held out, that gain falls to about 0.1 points, so on accuracy the model is
no better than always guessing "declining." ROC-AUC falls from 0.649 to 0.619, still above chance
(0.5), so a weak ranking signal remains. Values are means over three seeds, and per-client
results vary more between seeds, so they are directional. The baseline differs slightly between
splits because each split tests on different pages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
rows = []
for f in feats:
    aucs = []
    for seed in range(3):
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(X, y, g))
        m = make_model(seed).fit(X.iloc[tr][[f]], y.iloc[tr])
        aucs.append(roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te][[f]])[:, 1]))
    rows.append((f, round(sum(aucs) / len(aucs), 3)))
print('ROC-AUC of each feature alone, unseen clients (0.5 = chance):')
print(pd.DataFrame(rows, columns=['feature', 'auc']).to_string(index=False))

ROC-AUC of each feature alone, unseen clients (0.5 = chance):
       feature   auc
    imp_prev30 0.479
    pos_prev30 0.584
pos_std_prev30 0.526
      ctr_prev 0.606


## 3. Leakage audit

**Final feature set.** Four features, all measured in the earlier 30-day window, which does not
overlap the window that defines the label (`is_declining`: last-30-day impressions below 80% of
the previous 30 days): impressions (`imp_prev30`), average position (`pos_prev30`), position
volatility (`pos_std_prev30`) and click-through rate (`ctr_prev`).

**Test.** I scored each feature alone, on clients held out of training, with a small random
forest averaged over three seeds. A feature that leaks the label usually scores far above the
others on its own.

**Result (ROC-AUC, 0.5 = chance).** `imp_prev30` 0.479, `pos_prev30` 0.584,
`pos_std_prev30` 0.526, `ctr_prev` 0.606. No single feature scores suspiciously high. The
0.479 for impressions is below 0.5 but within the roughly 0.03 seed-to-seed noise I saw
elsewhere, so I read it as chance. The full four-feature model reached 0.619, only slightly
above the best single feature.

**What this does not show.** It checks for obvious single-feature leakage only. Position and
click-through rate are related (pages higher up tend to earn more clicks), so I did not separate
their effects. Features and label both come from the same table. Only the disjoint windows keep
them apart, and I did not test other time splits. Results are observed and directional.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

**My boldest sentence (first draft):** "The model predicts which pages will decline, so editors can
focus on the highest-risk pages."

**Why it overreaches.** The model does not forecast. In my capstone, the query features come
from a 90-day window that overlaps the 30 days that define decline, so they describe patterns
that appear together with a decline that has already happened. Even with features taken only
from the earlier window, the model did not beat a majority-class baseline on accuracy once whole
clients were held out (0.638 against 0.637), and ROC-AUC was 0.619, a weak ranking signal.

**Rewrite:** "Across held-out clients, pages whose visibility is concentrated in a single query
declined more often than other pages (about 68% against 60% for pages with no risk code). This is
an observed, directional association that can help an editor choose which pages to review first.
It is decision-support only: it is not a forecast, and it does not show that any pattern causes a
decline."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.